# Run GramVet ML in Google Colab

Upload the rebuilt package ZIP when prompted, then run cells top to bottom. Training uses five-fold CV and may take a few minutes. The dataset is synthetic; scores are not real-world clinical validation.

## 1. Upload and extract the package ZIP

Download `GramVet_ML_Package_Rebuilt.zip` to your computer first. The cell below opens Colab's upload picker.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()  # Choose GramVet_ML_Package_Rebuilt.zip
zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
PACKAGE_DIR = Path('/content/gramvet_ml')
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(PACKAGE_DIR)
print('Package extracted to:', PACKAGE_DIR)
print('Training script found:', (PACKAGE_DIR / 'models' / 'train_disease_model.py').exists())

## 2. Install dependencies

Colab already provides Jupyter and Python. This installs the package's listed Python libraries.

In [ ]:
%pip install -r {PACKAGE_DIR / 'requirements.txt'}

## 3. Generate the synthetic dataset

This creates 12,600 simulated rows. They are not patient or farm records.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(PACKAGE_DIR / 'generate_dataset.py'), '--rows-per-class', '1800', '--seed', '20260926'], cwd=PACKAGE_DIR, check=True)

## 4. Train, calibrate, and evaluate

The script benchmarks four models using five-fold cross-validation, fits calibration on the validation partition, and evaluates the held-out test partition.

In [ ]:
subprocess.run([sys.executable, str(PACKAGE_DIR / 'models' / 'train_disease_model.py')], cwd=PACKAGE_DIR, check=True)

## 5. View the evaluation results

In [ ]:
import json, pandas as pd
report = json.loads((PACKAGE_DIR / 'reports' / 'evaluation_report.json').read_text())
print('Selected model:', report['selected_model'])
display(pd.Series(report['test_final'], name='synthetic test metrics'))

## 6. Try a prediction

This uses one synthetic row as an example. For your own case, replace `sample_input` with a dictionary containing all 32 required input fields.

In [ ]:
import sys
sys.path.insert(0, str(PACKAGE_DIR))
from predict_disease import predict_disease
dataset = pd.read_csv(PACKAGE_DIR / 'data' / 'livestock_clinical_synthetic.csv')
sample_input = dataset.iloc[0].drop(labels='Disease').to_dict()
prediction = predict_disease(sample_input)
print(prediction)
print('Probability total:', sum(prediction['probabilities'].values()))

## 7. Download the trained package

This bundles the generated dataset, trained model, reports, and code into a ZIP and downloads it to your computer. Run this before the Colab runtime ends.

In [ ]:
import shutil
result_zip = Path('/content/GramVet_ML_Colab_results.zip')
shutil.make_archive(str(result_zip.with_suffix('')), 'zip', root_dir=PACKAGE_DIR)
files.download(str(result_zip))

The dataset is synthetic. Metrics and confidence are not real-world clinical validation; do not use the classifier as a standalone veterinary diagnosis.